In [2]:
import torch
import torch.nn as nn

In [48]:
class SelfAttention(nn.Module):
    def __init__(self, input_dim, inner_dim):
        super().__init__()

        self.inner_dim = inner_dim

        self.Wq = nn.Linear(input_dim, inner_dim)
        self.Wk = nn.Linear(input_dim, inner_dim)
        self.Wv = nn.Linear(input_dim, inner_dim)

        self.output_projection = nn.Linear(inner_dim, input_dim)

    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        attention = Q @ K.transpose(-1, -2)
        attention = attention / (self.inner_dim ** 0.5)
        attention = torch.softmax(attention, dim=-1)

        x = attention @ V
        x = self.output_projection(x)

        return x

In [87]:
class MultiheadAttention(nn.Module):
    def __init__(self, input_dim, inner_dim, h = 8):
        super().__init__()

        self.inner_dim = inner_dim
        self.input_dim = input_dim
        self.total_dim = self.inner_dim * h

        self.h = h

        self.Wq = nn.Linear(input_dim, self.total_dim)
        self.Wk = nn.Linear(input_dim, self.total_dim)
        self.Wv = nn.Linear(input_dim, self.total_dim)

        self.output_projection = nn.Linear(self.total_dim, self.input_dim)

    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        shape = Q.shape

        Q = Q.view(shape[0], shape[1], self.h, self.inner_dim)
        K = K.view(shape[0], shape[1], self.h, self.inner_dim)
        V = V.view(shape[0], shape[1], self.h, self.inner_dim)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)


        attention = Q @ K.transpose(-1, -2)
        attention = attention / (self.inner_dim ** 0.5)
        attention = torch.softmax(attention, dim=-1)

        x = attention @ V
        x = x.transpose(1, 2)
        x = x.reshape(shape[0], shape[1], self.total_dim)

        x = self.output_projection(x)

        return x

In [88]:
mhAttention = MultiheadAttention(128, 512)

In [51]:
transformer = SelfAttention(128, 512)

In [52]:
inp = torch.rand(16, 32, 128)

In [82]:
transformer(inp).shape

torch.Size([16, 32, 128])

In [89]:
mhAttention(inp).shape

torch.Size([16, 32, 128])